<a href="https://colab.research.google.com/github/v-yunusic/REPO-PYTHON-VY/blob/main/bank_additional_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PIP

In [1]:
pip install gdown

# LIBRERIAS

In [36]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st
import sympy as sp
import pandas as pd
import seaborn as sns
import re # RegEx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import gdown
import gc

#========= KAGGLE =========#
import kagglehub

#========= OS =========#
import os

print('Librerias importadas!\n'+'-'*60)

Librerias importadas!
------------------------------------------------------------


# PREPROCESAMIENTO

In [37]:
#IMPORTAR DATASET DESDE KAGGLE

path = kagglehub.dataset_download("sahistapatel96/bankadditionalfullcsv")
data_path = path+'/bank-additional-full.csv'

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bankadditionalfullcsv' dataset.
Path to dataset files: /kaggle/input/bankadditionalfullcsv


In [38]:
df = None

In [39]:
try:
  if not df or df.empty:
    df = pd.read_csv(data_path, sep=';')

    print('DATAFRAME CARGADO!\n')

except Exception as e:
  print('NO SE HA CARGADO DATAFRAME.')
  print(e)

DATAFRAME CARGADO!



In [40]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [41]:
print(f'Cantidad de filas:\n{df.shape[0]}')
print(f'Cantidad de columnas:\n{df.shape[1]}')

Cantidad de filas:
41188
Cantidad de columnas:
21


In [42]:
df = df.rename(columns={'age' : 'edad',
 'job' : 'trabajo',
 'marital' : 'e_civil',
 'education' : 'educacion',
 'default' : 'credito',
 'housing' : 'cred_hipotecario',
 'loan' : 'prestamo',
 'contact' : 'tipo_contacto',
 'month' : 'mes',
 'day_of_week' : 'dia_semana',
 'duration' : 'duracion',
 'campaign' : 'contactos_en_campaña',
 'pdays' : 'dias_contacto_prev',
 'previous' : 'contactos_previos',
 'poutcome' : 'resultado_anterior',
 'nr.employed' : 'num_empleados',
 'y' : 'suscribe',
 'emp.var.rate' : 'tasa.var.empleo',
 'cons.price.idx' : 'idx.precios.consumidor',
 'cons.conf.idx' : 'idx.confianza.consumidor'}, errors='raise')

In [43]:
print(f'Columnas presentes en el Dataframe:')
df.columns.tolist()

Columnas presentes en el Dataframe:


['edad',
 'trabajo',
 'e_civil',
 'educacion',
 'credito',
 'cred_hipotecario',
 'prestamo',
 'tipo_contacto',
 'mes',
 'dia_semana',
 'duracion',
 'contactos_en_campaña',
 'dias_contacto_prev',
 'contactos_previos',
 'resultado_anterior',
 'tasa.var.empleo',
 'idx.precios.consumidor',
 'idx.confianza.consumidor',
 'euribor3m',
 'num_empleados',
 'suscribe']

In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   edad                      41188 non-null  int64  
 1   trabajo                   41188 non-null  object 
 2   e_civil                   41188 non-null  object 
 3   educacion                 41188 non-null  object 
 4   credito                   41188 non-null  object 
 5   cred_hipotecario          41188 non-null  object 
 6   prestamo                  41188 non-null  object 
 7   tipo_contacto             41188 non-null  object 
 8   mes                       41188 non-null  object 
 9   dia_semana                41188 non-null  object 
 10  duracion                  41188 non-null  int64  
 11  contactos_en_campaña      41188 non-null  int64  
 12  dias_contacto_prev        41188 non-null  int64  
 13  contactos_previos         41188 non-null  int64  
 14  result

In [45]:
#CLASIFICACION DE COLUMNAS
perfil_cliente = df[[
  'edad',
  'trabajo',
  'e_civil',
  'educacion',
  'credito',
  'cred_hipotecario',
  'prestamo'
]]

cols_campaña = df[[
  'contactos_en_campaña',
  'dias_contacto_prev',
  'contactos_previos',
  'resultado_anterior',
  'tipo_contacto'
]]

macro_cols = df[[
  'tasa.var.empleo',
  'idx.precios.consumidor',
  'idx.confianza.consumidor',
  'euribor3m',
  'num_empleados'
]]

periodo_cols = df[[
    'mes',
    'dia_semana'
]]

var_objetivo = df['suscribe']
col_descarte = df['duracion'].copy()

df = df.drop(columns='duracion')

In [46]:
# DEFINICION DE COLUMNAS NUMERICAS Y CATEGORICAS
col_categoricas = df.select_dtypes(include=['object']).columns.tolist()
col_numericas = df.select_dtypes(exclude=['object','bool']).columns.tolist()
col_bools = df.select_dtypes(include=['bool']).columns.tolist()

print(f'COLUMNAS NUMERICAS ({len(col_numericas)}):\n{col_numericas}\n')
print(f'COLUMNAS CATEGORICAS ({len(col_categoricas)}):\n{col_categoricas}\n')
print(f'COLUMNAS BOOLS ({len(col_bools)}):\n{col_bools}\n')

COLUMNAS NUMERICAS (9):
['edad', 'contactos_en_campaña', 'dias_contacto_prev', 'contactos_previos', 'tasa.var.empleo', 'idx.precios.consumidor', 'idx.confianza.consumidor', 'euribor3m', 'num_empleados']

COLUMNAS CATEGORICAS (11):
['trabajo', 'e_civil', 'educacion', 'credito', 'cred_hipotecario', 'prestamo', 'tipo_contacto', 'mes', 'dia_semana', 'resultado_anterior', 'suscribe']

COLUMNAS BOOLS (0):
[]



In [47]:
for col in col_categoricas:
  valores = df[col].unique().tolist()
  print(f'{col} ({len(valores)}): \n{valores}\n')

trabajo (12): 
['housemaid', 'services', 'admin.', 'blue-collar', 'technician', 'retired', 'management', 'unemployed', 'self-employed', 'unknown', 'entrepreneur', 'student']

e_civil (4): 
['married', 'single', 'divorced', 'unknown']

educacion (8): 
['basic.4y', 'high.school', 'basic.6y', 'basic.9y', 'professional.course', 'unknown', 'university.degree', 'illiterate']

credito (3): 
['no', 'unknown', 'yes']

cred_hipotecario (3): 
['no', 'yes', 'unknown']

prestamo (3): 
['no', 'yes', 'unknown']

tipo_contacto (2): 
['telephone', 'cellular']

mes (10): 
['may', 'jun', 'jul', 'aug', 'oct', 'nov', 'dec', 'mar', 'apr', 'sep']

dia_semana (5): 
['mon', 'tue', 'wed', 'thu', 'fri']

resultado_anterior (3): 
['nonexistent', 'failure', 'success']

suscribe (2): 
['no', 'yes']



In [48]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct": df.isna().mean() * 100,
}).sort_values("pct", ascending=False)

nulos

,nulos,pct
edad,0,0.0
trabajo,0,0.0
e_civil,0,0.0
educacion,0,0.0
credito,0,0.0
cred_hipotecario,0,0.0
prestamo,0,0.0
tipo_contacto,0,0.0
mes,0,0.0
dia_semana,0,0.0


In [49]:
df[col_categoricas].describe()

,trabajo,e_civil,educacion,credito,cred_hipotecario,prestamo,tipo_contacto,mes,dia_semana,resultado_anterior,suscribe
count,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188
unique,12,4,8,3,3,3,2,10,5,3,2
top,admin.,married,university.degree,no,yes,no,cellular,may,thu,nonexistent,no
freq,10422,24928,12168,32588,21576,33950,26144,13769,8623,35563,36548


In [50]:
df.loc[df.duplicated()]

,edad,trabajo,e_civil,educacion,credito,cred_hipotecario,prestamo,tipo_contacto,mes,dia_semana,contactos_en_campaña,dias_contacto_prev,contactos_previos,resultado_anterior,tasa.var.empleo,idx.precios.consumidor,idx.confianza.consumidor,euribor3m,num_empleados,suscribe
10,41,blue-collar,married,unknown,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
11,25,services,single,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
16,35,blue-collar,married,basic.6y,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
31,59,technician,married,unknown,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
104,52,admin.,divorced,university.degree,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39985,27,admin.,single,high.school,no,no,no,cellular,jun,tue,2,999,0,nonexistent,-1.7,94.055,-39.8,0.761,4991.6,yes
40401,31,student,single,unknown,no,yes,no,cellular,aug,thu,2,999,0,nonexistent,-1.7,94.027,-38.3,0.904,4991.6,yes
40404,41,entrepreneur,married,university.degree,no,yes,no,cellular,aug,thu,1,999,0,nonexistent,-1.7,94.027,-38.3,0.904,4991.6,yes
40806,35,technician,married,professional.course,no,yes,no,cellular,sep,thu,1,999,2,failure,-1.1,94.199,-37.5,0.878,4963.6,no


In [51]:
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

In [74]:
rangos = [0, 17, 30, 50, 65, 100]
labels = ['Menor de edad', 'Jóvenes (18-30)', 'Adultos (31-50)', 'Adultos Mayores (51-65)', 'Adultos de la Tercera Edad (>65)']

df['rango_etario'] = pd.cut(df['edad'], bins=rangos, labels=labels, right=True)

df['rango_etario'].value_counts()

,count
rango_etario,
Adultos (31-50),25424
Jóvenes (18-30),6997
Adultos Mayores (51-65),6372
Adultos de la Tercera Edad (>65),606
Menor de edad,5


In [75]:
df.loc[df['rango_etario'] == 'Menor de edad']

,edad,trabajo,e_civil,educacion,credito,cred_hipotecario,prestamo,tipo_contacto,mes,dia_semana,...,dias_contacto_prev,contactos_previos,resultado_anterior,tasa.var.empleo,idx.precios.consumidor,idx.confianza.consumidor,euribor3m,num_empleados,suscribe,rango_etario
37140,17,student,single,unknown,no,yes,no,cellular,aug,wed,...,4,2,success,-2.9,92.201,-31.4,0.884,5076.2,no,Menor de edad
37539,17,student,single,basic.9y,no,yes,no,cellular,aug,fri,...,999,2,failure,-2.9,92.201,-31.4,0.869,5076.2,no,Menor de edad
37558,17,student,single,basic.9y,no,yes,no,cellular,aug,fri,...,4,2,success,-2.9,92.201,-31.4,0.869,5076.2,no,Menor de edad
37579,17,student,single,basic.9y,no,unknown,unknown,cellular,aug,fri,...,999,1,failure,-2.9,92.201,-31.4,0.869,5076.2,yes,Menor de edad
38274,17,student,single,unknown,no,no,yes,cellular,oct,tue,...,2,2,success,-3.4,92.431,-26.9,0.742,5017.5,yes,Menor de edad


In [77]:
a = df.groupby('rango_etario')['suscribe']

a.value_counts()

/tmp/ipykernel_843/3534984703.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  a = df.groupby('rango_etario')['suscribe']


rango_etario                      suscribe
Menor de edad                     no              3
                                  yes             2
Jóvenes (18-30)                   no           5890
                                  yes          1107
Adultos (31-50)                   no          23009
                                  yes          2415
Adultos Mayores (51-65)           no           5583
                                  yes           789
Adultos de la Tercera Edad (>65)  no            321
                                  yes           285
Name: count, dtype: int64

In [78]:
b = df.groupby('trabajo')['suscribe']
b.value_counts()

trabajo        suscribe
admin.         no          8538
               yes         1335
blue-collar    no          8202
               yes          633
entrepreneur   no          1284
               yes          121
housemaid      no           922
               yes          106
management     no          2494
               yes          326
retired        no          1252
               yes          431
self-employed  no          1237
               yes          149
services       no          3479
               yes          322
student        no           581
               yes          271
technician     no          5677
               yes          727
unemployed     no           851
               yes          141
unknown        no           289
               yes           36
Name: count, dtype: int64